In [9]:
!pip install rdflib pandas tabulate requests google-genai

In [10]:
import requests

base_url = "https://raw.githubusercontent.com/estebanarredondov/tarea_web_semantica_datos_abiertos/main"

archivos = {
    "rentas_vitalicias_validos_30.ttl": f"{base_url}/datos/rentas_vitalicias_validos_30.ttl",
    "rentas_vitalicias_shacl.ttl": f"{base_url}/esquemas/rentas_vitalicias_shacl.ttl",
}

for nombre, url in archivos.items():
    r = requests.get(url)
    print(url, r.status_code)  # 👈 debug
    r.raise_for_status()
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(r.text)

print("archivos descargados")

https://raw.githubusercontent.com/estebanarredondov/tarea_web_semantica_datos_abiertos/main/datos/rentas_vitalicias_validos_30.ttl 200
https://raw.githubusercontent.com/estebanarredondov/tarea_web_semantica_datos_abiertos/main/esquemas/rentas_vitalicias_shacl.ttl 200
archivos descargados


In [11]:
from rdflib import Graph

archivo_esquema = "rentas_vitalicias_shacl.ttl"
archivo_datos = "rentas_vitalicias_validos_30.ttl"

print("Usando archivos:")
print("Esquema:", archivo_esquema)
print("Datos:", archivo_datos)

# Cargar grafo
g = Graph()
g.parse(archivo_esquema, format="turtle")
g.parse(archivo_datos, format="turtle")

print("\nTripletas totales:", len(g))

Usando archivos:
Esquema: rentas_vitalicias_shacl.ttl
Datos: rentas_vitalicias_validos_30.ttl

Tripletas totales: 307


In [12]:
import pandas as pd

def ejecutar_query(graph, query):
    resultados = graph.query(query)
    filas = []
    for row in resultados:
        fila = {str(var): str(val) for var, val in row.asdict().items()}
        filas.append(fila)
    return pd.DataFrame(filas)

In [13]:
URI_CLASE_PENSIONADO = "http://example.com/rentas#Pensionado"
URI_PROP_MONTO = "http://example.com/rentas#pension"
URI_PROP_TIPO = "http://example.com/rentas#tipoPension"
URI_PROP_MODALIDAD = "http://example.com/rentas#modalidad"
URI_PROP_SEXO = "http://example.com/rentas#sexo"
URI_PENSIONADO_EJEMPLO = "http://example.com/rentas#Cliente14"

preguntas_trabajo = {
    "P1": {
        "texto": "¿Cuál es el promedio de pensión por tipo?",
        "query": f"""
        SELECT ?tipo (AVG(xsd:decimal(?monto)) AS ?promedio)
        WHERE {{
          ?p a <{URI_CLASE_PENSIONADO}> ;
             <{URI_PROP_TIPO}> ?tipo ;
             <{URI_PROP_MONTO}> ?monto .
        }}
        GROUP BY ?tipo
        ORDER BY ?tipo
        """,
        "label": "promedio_tipo"
    },

    "P2": {
        "texto": "¿Cuál es el promedio de pensión por modalidad?",
        "query": f"""
        SELECT ?modalidad (AVG(xsd:decimal(?monto)) AS ?promedio)
        WHERE {{
          ?p a <{URI_CLASE_PENSIONADO}> ;
             <{URI_PROP_MODALIDAD}> ?modalidad ;
             <{URI_PROP_MONTO}> ?monto .
        }}
        GROUP BY ?modalidad
        ORDER BY ?modalidad
        """,
        "label": "promedio_modalidad"
    },

    "P3": {
        "texto": "¿Qué modalidad tiene mayores montos?",
        "query": f"""
        SELECT ?modalidad (AVG(xsd:decimal(?monto)) AS ?promedio)
        WHERE {{
          ?p a <{URI_CLASE_PENSIONADO}> ;
             <{URI_PROP_MODALIDAD}> ?modalidad ;
             <{URI_PROP_MONTO}> ?monto .
        }}
        GROUP BY ?modalidad
        ORDER BY DESC(?promedio)
        LIMIT 1
        """,
        "label": "mejor_modalidad"
    },

    "P4": {
        "texto": "¿Cuál es el promedio de pensión por sexo?",
        "query": f"""
        SELECT ?sexo (AVG(xsd:decimal(?monto)) AS ?promedio)
        WHERE {{
          ?p a <{URI_CLASE_PENSIONADO}> ;
             <{URI_PROP_SEXO}> ?sexo ;
             <{URI_PROP_MONTO}> ?monto .
        }}
        GROUP BY ?sexo
        ORDER BY ?sexo
        """,
        "label": "promedio_sexo"
    },

    "P5": {
        "texto": "¿Este pensionado está sobre o bajo el promedio del mercado?",
        "query": f"""
        SELECT ?montoIndividual (AVG(xsd:decimal(?montoMercado)) AS ?promedioMercado)
        WHERE {{
          BIND(<{URI_PENSIONADO_EJEMPLO}> AS ?pensionadoIndividual) . # This will be replaced dynamically
          ?pensionadoIndividual <{URI_PROP_MONTO}> ?montoIndividual_dummy . # Placeholder for dynamic value
          ?p a <{URI_CLASE_PENSIONADO}> ;
             <{URI_PROP_MONTO}> ?montoMercado .
        }}
        """,
        "label": "comparacion_mercado"
    }
}

In [14]:
for codigo, info in preguntas_trabajo.items():
    print(f"\n{'='*60}")
    print(codigo, "-", info["texto"])
    print(f"{'='*60}")

    try:
        df = ejecutar_query(g, info["query"])
        print(df.to_markdown(index=False))
    except Exception as e:
        print("Error al ejecutar la consulta:", e)


P1 - ¿Cuál es el promedio de pensión por tipo?
| tipo          |   promedio |
|:--------------|-----------:|
| Invalidez     |    15.7143 |
| Sobrevivencia |     7.1    |
| Vejez         |    10.0167 |

P2 - ¿Cuál es el promedio de pensión por modalidad?
| modalidad                                          |   promedio |
|:---------------------------------------------------|-----------:|
| http://example.com/rentas#RentaVitalicia           |    11.8688 |
| http://example.com/rentas#RentaVitaliciaEscalonada |    10.25   |
| http://example.com/rentas#RetiroProgramado         |     9.3    |

P3 - ¿Qué modalidad tiene mayores montos?
| modalidad                                |   promedio |
|:-----------------------------------------|-----------:|
| http://example.com/rentas#RentaVitalicia |    11.8688 |

P4 - ¿Cuál es el promedio de pensión por sexo?
| sexo   |   promedio |
|:-------|-----------:|
| Hombre |   12.7267  |
| Mujer  |    8.99333 |

P5 - ¿Este pensionado está sobre o bajo el

In [15]:
import os
from getpass import getpass
from google import genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')

client = genai.Client(api_key=api_key)

In [16]:

def clasificar_pregunta(pregunta, model):
    prompt = f"""
Clasifica la pregunta del usuario en una de estas categorías.
Si la pregunta es de tipo 'comparacion_mercado' y contiene un valor numérico de pensión, responde con la etiqueta seguida de ':' y el valor (ej. comparacion_mercado:9.10).
Para otras preguntas, responde SOLO con una etiqueta exacta.

Etiquetas:
- promedio_tipo
- promedio_modalidad
- mejor_modalidad
- promedio_sexo
- comparacion_mercado

Pregunta:
{pregunta}
""".strip()

    response = client.models.generate_content(
        model=model,
        contents=prompt
    )

    full_response = response.text.strip()
    label = None
    pension_value = None

    if ':' in full_response:
        parts = full_response.split(':', 1)
        label = parts[0].strip()
        try:
            pension_value = float(parts[1].strip())
        except ValueError:
            pension_value = None
    else:
        label = full_response

    return label, pension_value

def obtener_contexto(clave, df):
    # Find the corresponding question text from preguntas_trabajo
    original_question_text = ""
    for codigo, info in preguntas_trabajo.items():
        if "label" in info and info["label"] == clave:
            original_question_text = info["texto"]
            break

    output_lines = []
    if original_question_text:
        output_lines.append(f"Pregunta: {original_question_text}")

    if df is None or df.empty:
        output_lines.append("No se encontraron resultados en el grafo.")
        return "\n".join(output_lines)

    if clave == "promedio_tipo":
        output_lines.append("Promedios de pensión por tipo:")
        for _, row in df.iterrows():
            output_lines.append(f"- Tipo: {row['tipo']} | Promedio: {row['promedio']}")

    elif clave == "promedio_modalidad":
        output_lines.append("Promedios de pensión por modalidad:")
        for _, row in df.iterrows():
            output_lines.append(f"- Modalidad: {row['modalidad']} | Promedio: {row['promedio']}")

    elif clave == "mejor_modalidad":
        row = df.iloc[0]
        output_lines.append("Resultado de comparación de modalidades:")
        output_lines.append(f"- Modalidad con mayor promedio: {row['modalidad']}")
        output_lines.append(f"- Promedio: {row['promedio']}")

    elif clave == "promedio_sexo":
        output_lines.append("Promedios de pensión por sexo:")
        for _, row in df.iterrows():
            output_lines.append(f"- Sexo: {row['sexo']} | Promedio: {row['promedio']}")

    elif clave == "comparacion_mercado":
        row = df.iloc[0]
        output_lines.append("Comparación entre pensionado individual y mercado:")
        output_lines.append(f"- Monto individual: {row['montoIndividual']}")
        output_lines.append(f"- Promedio del mercado: {row['promedioMercado']}")
    else:
        output_lines.append("No se pudo construir contexto.")

    return "\n".join(output_lines)

def responder(pregunta, contexto, model):
    prompt = f"""
Eres un asistente que responde preguntas sobre pensiones en Chile.

Responde SOLO usando la información del contexto.
No inventes datos.
Si el contexto no es suficiente, dilo explícitamente.
No agregues información externa.

Contexto:
{contexto}

Pregunta:
{pregunta}

Entrega una respuesta clara, breve y en español.
""".strip()

    response = client.models.generate_content(
        model=model,
        contents=prompt
    )

    return response.text


In [17]:
def pipeline(graph, pregunta, model):
    label, pension_value = clasificar_pregunta(pregunta, model)
    if label is None:
        return {
            "error": "No pude asociar la pregunta con una consulta del grafo.",
            "df": None,
            "contexto": None,
            "respuesta": None
        }

    # Get the query corresponding to the classified key
    query_info = None
    for codigo, info in preguntas_trabajo.items():
        if "label" in info and info["label"] == label:
            query_info = info
            break

    if query_info is None:
        return {
            "error": f"No se encontró la consulta para la clave: {label}",
            "df": None,
            "contexto": None,
            "respuesta": None
        }

    # Dynamic query modification for 'comparacion_mercado'
    if label == "comparacion_mercado" and pension_value is not None:
        original_query = query_info["query"]

        # Define the lines to be replaced/removed based on constants
        search_bind_pensionado_line = f"BIND(<{URI_PENSIONADO_EJEMPLO}> AS ?pensionadoIndividual) ."
        search_monto_dummy_line = f"?pensionadoIndividual <{URI_PROP_MONTO}> ?montoIndividual_dummy ."

        # Create the replacement BIND line for the individual's pension value
        replacement_bind_monto_line = f"BIND({pension_value} AS ?montoIndividual) ."

        # Perform the replacements
        # Replace the specific BIND line for URI_PENSIONADO_EJEMPLO
        modified_query = original_query.replace(search_bind_pensionado_line, replacement_bind_monto_line)
        # Remove the placeholder line for ?montoIndividual_dummy
        modified_query = modified_query.replace(search_monto_dummy_line, "")

        # Clean up any potential double newlines if a line was removed
        modified_query = "\n".join([line for line in modified_query.split('\n') if line.strip()])

        # Assign the modified query back to query_info for execution
        query_info["query"] = modified_query

    try:
        df_resultado = ejecutar_query(graph, query_info["query"])
    except Exception as e:
        return {
            "error": f"Error al ejecutar la consulta para '{label}': {e}",
            "df": None,
            "contexto": None,
            "respuesta": None
        }

    contexto = obtener_contexto(label, df_resultado)
    respuesta = responder(pregunta, contexto, model)

    return {
        "error": None,
        "df": df_resultado,
        "contexto": contexto,
        "respuesta": respuesta
    }


In [18]:
pregunta = "¿Qué modalidad tiene mayores montos?"

resultado = pipeline(g, pregunta, "gemini-2.5-flash")

if resultado["error"]:
    print(resultado["error"])
else:
    print("=== RESULTADO ESTRUCTURADO ===")
    display(resultado["df"])

    print("\n=== CONTEXTO ENVIADO A GEMINI ===")
    print(resultado["contexto"])

    print("\n=== RESPUESTA DE GEMINI ===")
    print(resultado["respuesta"])

import time

print("Esperando...")
time.sleep(1)
print("Listo")

=== RESULTADO ESTRUCTURADO ===


,modalidad,promedio
0,http://example.com/rentas#RentaVitalicia,11.86875



=== CONTEXTO ENVIADO A GEMINI ===
Pregunta: ¿Qué modalidad tiene mayores montos?
Resultado de comparación de modalidades:
- Modalidad con mayor promedio: http://example.com/rentas#RentaVitalicia
- Promedio: 11.86875

=== RESPUESTA DE GEMINI ===
La modalidad con mayores montos, según el promedio, es la Renta Vitalicia.
Esperando...
Listo


In [19]:
pregunta = "¿compara La pensión 9.10 está sobre o bajo el promedio del mercado?"

resultado = pipeline(g, pregunta, "gemini-2.5-flash")

if resultado["error"]:
    print(resultado["error"])
else:
    print("=== RESULTADO ESTRUCTURADO ===")
    display(resultado["df"])

    print("\n=== CONTEXTO ENVIADO A GEMINI ===")
    print(resultado["contexto"])

    print("\n=== RESPUESTA DE GEMINI ===")
    print(resultado["respuesta"])

import time

print("Esperando...")
time.sleep(1)
print("Listo")

=== RESULTADO ESTRUCTURADO ===


,montoIndividual,promedioMercado
0,9.1,10.86



=== CONTEXTO ENVIADO A GEMINI ===
Pregunta: ¿Este pensionado está sobre o bajo el promedio del mercado?
Comparación entre pensionado individual y mercado:
- Monto individual: 9.1
- Promedio del mercado: 10.86

=== RESPUESTA DE GEMINI ===
La pensión de 9.10 está bajo el promedio del mercado (10.86).
Esperando...
Listo


In [20]:
pregunta = "¿Cuál es el promedio de pensión por tipo?"

resultado = pipeline(g, pregunta, "gemini-2.5-flash")

if resultado["error"]:
    print(resultado["error"])
else:
    print("=== RESULTADO ESTRUCTURADO ===")
    display(resultado["df"])

    print("\n=== CONTEXTO ENVIADO A GEMINI ===")
    print(resultado["contexto"])

    print("\n=== RESPUESTA DE GEMINI ===")
    print(resultado["respuesta"])

import time

print("Esperando...")
time.sleep(1)
print("Listo")

=== RESULTADO ESTRUCTURADO ===


,tipo,promedio
0,Invalidez,15.71428571428571428571428571
1,Sobrevivencia,7.1
2,Vejez,10.01666666666666666666666667



=== CONTEXTO ENVIADO A GEMINI ===
Pregunta: ¿Cuál es el promedio de pensión por tipo?
Promedios de pensión por tipo:
- Tipo: Invalidez | Promedio: 15.71428571428571428571428571
- Tipo: Sobrevivencia | Promedio: 7.1
- Tipo: Vejez | Promedio: 10.01666666666666666666666667

=== RESPUESTA DE GEMINI ===
Los promedios de pensión por tipo son:

*   **Invalidez:** 15.71428571428571428571428571
*   **Sobrevivencia:** 7.1
*   **Vejez:** 10.01666666666666666666666667
Esperando...
Listo


In [21]:
pregunta = "¿Cuál es el promedio de pensión por tipo?"

resultado = pipeline(g, pregunta, "gemini-2.5-flash")

if resultado["error"]:
    print(resultado["error"])
else:
    print("=== RESULTADO ESTRUCTURADO ===")
    display(resultado["df"])

    print("\n=== CONTEXTO ENVIADO A GEMINI ===")
    print(resultado["contexto"])

    print("\n=== RESPUESTA DE GEMINI ===")
    print(resultado["respuesta"])

import time

print("Esperando...")
time.sleep(1)
print("Listo")

=== RESULTADO ESTRUCTURADO ===


,tipo,promedio
0,Invalidez,15.71428571428571428571428571
1,Sobrevivencia,7.1
2,Vejez,10.01666666666666666666666667



=== CONTEXTO ENVIADO A GEMINI ===
Pregunta: ¿Cuál es el promedio de pensión por tipo?
Promedios de pensión por tipo:
- Tipo: Invalidez | Promedio: 15.71428571428571428571428571
- Tipo: Sobrevivencia | Promedio: 7.1
- Tipo: Vejez | Promedio: 10.01666666666666666666666667

=== RESPUESTA DE GEMINI ===
Los promedios de pensión por tipo son:
*   **Invalidez:** 15.71428571428571428571428571
*   **Sobrevivencia:** 7.1
*   **Vejez:** 10.01666666666666666666666667
Esperando...
Listo


In [22]:
pregunta = "¿Cuál es el promedio de pensión por sexo?"

resultado = pipeline(g, pregunta, "gemini-2.5-flash")

if resultado["error"]:
    print(resultado["error"])
else:
    print("=== RESULTADO ESTRUCTURADO ===")
    display(resultado["df"])

    print("\n=== CONTEXTO ENVIADO A GEMINI ===")
    print(resultado["contexto"])

    print("\n=== RESPUESTA DE GEMINI ===")
    print(resultado["respuesta"])

import time

print("Esperando...")
time.sleep(1)
print("Listo")

=== RESULTADO ESTRUCTURADO ===


,sexo,promedio
0,Hombre,12.72666666666666666666666667
1,Mujer,8.993333333333333333333333333



=== CONTEXTO ENVIADO A GEMINI ===
Pregunta: ¿Cuál es el promedio de pensión por sexo?
Promedios de pensión por sexo:
- Sexo: Hombre | Promedio: 12.72666666666666666666666667
- Sexo: Mujer | Promedio: 8.993333333333333333333333333

=== RESPUESTA DE GEMINI ===
El promedio de pensión por sexo es:
*   **Hombre:** 12.72666666666666666666666667
*   **Mujer:** 8.993333333333333333333333333
Esperando...
Listo
